# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features 
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Imports

In [1]:
#Imports
import os
import numpy as np
import pandas as pd
import sqlite3
#plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

#import joypy
from scipy import stats

from plate_information import *
from plate_preprocessing import *


## Get table info for a single plate

# Exporting zone: 
Make sure you put "_active" in the CP output folder names!

In [2]:
# Setting file paths
curr_plates = ["20240313_rep01","20240326_rep02","20241018_rep03", "20241112_rep04", "20250328_rep05", "20250410_rep06", "20250501_rep07"] #"20240313_rep01_output","20240326_rep02_output"
parent_dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output"

parent_object_table = "Per_Cell"
compartment_tables = [ 'Per_MergedNucleiPerCell','Per_MergedMitoPerCell','Per_MergedLysoPerCell'] #note that these must be in a 1:1 relationship with cell

# query designed to remove cells from the dataset without any mitochondria, nuclei or lysosomes; allows us to have a 1:1 relationship
parent_obj_query = f"SELECT * FROM Per_{parent_object_table} WHERE Cell_Children_Mitochondria_Count > 0 AND Cell_Children_Lysosomes_Count > 0 AND Cell_Children_Nuclei_Count > 0;"

# Initialize a list to store the combined DataFrames
plate_dfs = {}


In [3]:
def load_and_combine_plates_from_db(
    parent_dir,
    curr_plates,
    compartment_name="Cell",
    combine_dfs=True,
    calculate_medians=True,
):
    # Loop over the plates
    for root, dirs, files in os.walk(parent_dir):
        for filename in files:
            if (
                filename.endswith(".db")
                and "active" in root
                and "extra" not in filename
                and "output" in filename
            ):
                db_path = os.path.join(root, filename)  # make the path
                for plate in curr_plates:
                    if plate in db_path:
                        conn = sqlite3.connect(db_path)
                        cursor = conn.cursor()
                        # update_database_with_well_metadata(db_path)
                        try:
                            # Read the 'Per_Cell' table and get metadata from 'Per_Image' table
                            pre_cell_df = pd.read_sql_query(
                                f"SELECT * FROM Per_{compartment_name}", conn
                            )
                            image_df = pd.read_sql_query(
                                "SELECT * FROM Per_Image", conn
                            )
                            # metadata extraction - make sure its the exact same file format as the one above
                            map_file = os.path.join(
                                "plate_metadata", f"{plate}_metadata", "map.csv"
                            )
                            print(map_file)

                            cell_df = pre_cell_df.merge(
                                image_df, on=["ImageNumber"], how="left"
                            )
                            cell_df.columns = cell_df.columns.str.replace(
                                r"^Image_Metadata_", "Metadata_", regex=True
                            )

                            # merge dfs together that are 1:1 e.g nuc and cyto with cell
                            if combine_dfs:
                                cell_df = combine_one_to_one_dfs(cell_df, conn)

                            # remove rows where there isn't a valid row/column/field metadata
                            cols_to_check = [
                                "Metadata_WellRow",
                                "Metadata_WellColumn",
                                "Metadata_Field",
                            ]
                            cell_df = cell_df.replace(
                                [np.inf, -np.inf], np.nan
                            )  # Replace inf with NaN
                            cell_df = cell_df.dropna(
                                subset=cols_to_check
                            )  # Drop rows with NaN in these columns

                            # make these metadatas int
                            cell_df["Metadata_WellRow"] = cell_df[
                                "Metadata_WellRow"
                            ].astype(int)
                            cell_df["Metadata_WellColumn"] = cell_df[
                                "Metadata_WellColumn"
                            ].astype(int)
                            cell_df["Metadata_Field"] = cell_df[
                                "Metadata_Field"
                            ].astype(int)

                            # add the median data
                            if calculate_medians:
                                extra_feature_filename = "extra_features.db"
                                extra_feature_db_path = os.path.join(
                                    root, extra_feature_filename
                                )
                                cell_df = load_organelle_medians(
                                    db_path=extra_feature_db_path, df=cell_df
                                )
                                cell_df.reset_index(drop=True)

                            # Find cell/nuc area ratio
                            cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(
                                cell_df,
                                cell_area_col="Cell_AreaShape_Area",
                                nuc_area_col="Nuclei_AreaShape_Area",
                            )
                            # get rid of cells where area is lower than nuc area
                            cell_df = cell_df[
                                cell_df["Cell_Nuclei_Area_Ratio"] > 1
                            ].reset_index(drop=True)

                            print(os.path.exists(map_file))

                            if os.path.exists(map_file):
                                platemap_df = pd.read_csv(map_file)
                                #display(platemap_df)
                                platemap_df["Metadata_WellRow"] = platemap_df[
                                    "Metadata_WellRow"
                                ].astype(int)
                                platemap_df["Metadata_WellColumn"] = platemap_df[
                                    "Metadata_WellColumn"
                                ].astype(int)
                                platemap_df["Metadata_Field"] = platemap_df[
                                    "Metadata_Field"
                                ].astype(int)

                                # platemap_df.reset_index(drop=True)
                                cell_df = cell_df.merge(
                                    platemap_df,
                                    on=[
                                        "Metadata_WellRow",
                                        "Metadata_WellColumn",
                                        "Metadata_Field",
                                        "Metadata_Well"
                                    ],
                                    how="left",
                                )
                                # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
                                cell_df["Passage Group"] = cell_df[
                                    "PassageNumber"
                                ].apply(passage_group)
                                cell_df["AllGroups"] = add_drug_to_group(
                                    cell_df, "Passage Group", "Drug"
                                )
                                
                                #additional metadatas
                                cell_df["Metadata_WellRowColumnField"] = (
                                    "r"
                                    + cell_df["Metadata_WellRow"].astype(str)
                                    + "c"
                                    + cell_df["Metadata_WellColumn"].astype(str)
                                    + "f"
                                    + cell_df["Metadata_Field"].astype(str)
                                )

                                cell_df["Metadata_Plate"] = plate
                                cell_df["Replicate_Number"] = plate[-1]
                                cell_df["Replicate_String"] = f"R{plate[-1]}"
                                
                                cell_df["Replicate_WellRowColumnField"] = cell_df["Replicate_String"] + "_" + cell_df["Metadata_WellRowColumnField"]

                            plate_dfs[plate] = cell_df
                        except Exception as e:
                            print(f"Error reading {db_path}: {e}")
                        finally:
                            conn.close()

    # Combine all DataFrames
    combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
    return combined_cell_df


In [4]:
combined_cell_df = load_and_combine_plates_from_db(parent_dir,curr_plates)
#combined_nuclei_df = pd.concat(nuclei_dfs.values(), ignore_index=True)
                
#Filter DataFrames to only include cells that were stained with LAMP1-488 and MitoRed



plate_metadata/20250410_rep06_metadata/map.csv
True
plate_metadata/20240326_rep02_metadata/map.csv
True
plate_metadata/20250501_rep07_metadata/map.csv
True
plate_metadata/20241112_rep04_metadata/map.csv
True
plate_metadata/20250328_rep05_metadata/map.csv
True
plate_metadata/20241018_rep03_metadata/map.csv
True
plate_metadata/20240313_rep01_metadata/map.csv
True


## Export everything to CSV

In [ ]:
#Export to a giant csv
# filter out the non-experimental test images
combined_cell_df_mitolyso = combined_cell_df[combined_cell_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]

display(combined_cell_df_mitolyso.head(10))
# print(cell_df.shape, " ", filter_df.shape)

#export the plate
outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"
combined_cell_df_mitolyso.to_csv(
    os.path.join(outpath, "total_combined_cell.csv"), index=False
)

#set up the image borders and make a csv with excluded border cells
min_x = 0
min_y = 0
max_x = combined_cell_df_mitolyso["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = combined_cell_df_mitolyso["Image_Height_DAPI"][0]

combined_cell_df_mitolyso_borders_excluded = exclude_borders(
    combined_cell_df_mitolyso, min_x, min_y, max_x, max_y, prefix="Cell_"
)
combined_cell_df_mitolyso_borders_excluded.to_csv(
    os.path.join(outpath, "total_combined_cell_borders_excluded.csv"), index=False
)


,ImageNumber,Cell_Number_Object_Number,Cell_AreaShape_Area,Cell_AreaShape_BoundingBoxArea,Cell_AreaShape_BoundingBoxMaximum_X,Cell_AreaShape_BoundingBoxMaximum_Y,Cell_AreaShape_BoundingBoxMinimum_X,Cell_AreaShape_BoundingBoxMinimum_Y,Cell_AreaShape_Center_X,Cell_AreaShape_Center_Y,...,Image_ExecutionTime_72MeasureTexture,Image_ExecutionTime_73MeasureColocalization,Image_ExecutionTime_74CalculateMoments,Image_ExecutionTime_75CalculateMoments,Image_ExecutionTime_76RelateObjects,Image_ExecutionTime_79RelateObjects,Image_ExecutionTime_94SaveImages,Image_ExecutionTime_95SaveImages,Image_ExecutionTime_77RelateObjects,Image_ExecutionTime_93SaveImages
0,93,1,576629.0,2533020.0,2137.0,1933.0,507.0,379.0,1592.324406,1271.878466,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,95,1,160855.0,250275.0,1032.0,355.0,327.0,0.0,679.381014,179.907998,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100,1,337955.0,721344.0,816.0,2160.0,0.0,1276.0,359.044574,1723.967836,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,99,1,256708.0,762552.0,1921.0,1997.0,1445.0,395.0,1658.952810,873.709756,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,98,1,477796.0,2658292.0,1757.0,2105.0,15.0,579.0,957.668377,1285.416425,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,98,2,116904.0,223500.0,1168.0,2017.0,423.0,1717.0,759.568193,1883.020222,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,97,1,982750.0,1945048.0,2153.0,1316.0,675.0,0.0,1270.548413,410.023866,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,97,2,241819.0,365210.0,295.0,1877.0,0.0,639.0,153.799226,1334.838139,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,105,1,277600.0,1659452.0,1441.0,1441.0,287.0,3.0,1032.353883,654.249878,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,105,2,95696.0,156996.0,623.0,443.0,0.0,191.0,246.535111,320.736081,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
display(combined_cell_df_mitolyso[["Cell_Number_Object_Number", "Nuclei_Number_Object_Number","Cell_Parent_Nuclei"]].head(10))


,Cell_Number_Object_Number,Nuclei_Number_Object_Number,Cell_Parent_Nuclei
0,1,1,1
1,1,1,1
2,1,1,1
3,1,1,1
4,1,1,1
5,2,2,2
6,1,1,1
7,2,2,2
8,1,2,2
9,2,1,1


### Summary Stats

In [ ]:
def passage_groups_sort_key(group_name):
    """
    Key function for natural sorting of strings containing numbers.
    Extract numeric parts and convert to int .
    """
    digit_pattern = r"([0-9]+)"  # Matches "RX" where X is the replicate number (placeholder for now)
    match = re.search(digit_pattern, group_name)
    if match:
        first_digit = int(match.group(1))
        return first_digit
    else:
        text = group_name.lower()
        if text == "doxo":
            return 999
        else:
            return ValueError


def make_summary_stats_for_df_and_feature(
    df,
    x_value,
    feature,
    summary_outpath,
    df_tag="original",
    replicate_col_name="Replicate_Number",
    feature_name="area",
    group_name="passage_group",
    include_cols=[],
):
    from pathlib import Path
    try:
        table_csvname = f"{df_tag}_total_combined_{feature_name}_stats.csv"
        feature_csvname = f"{df_tag}_{feature_name}_by_{group_name}_stats.csv"
        agg_feature_csvname = f"{df_tag}_agg_{feature_name}_by_{group_name}_stats.csv"

        subfolder_name = f"{df_tag}_{feature_name}_summary_stats"
        parent_folder = Path(summary_outpath, subfolder_name)
        parent_folder.mkdir(exist_ok=True)

        if not include_cols:
            df_to_summarize = df
        else:
            df_to_summarize = df[include_cols]
        df_to_summarize.describe().to_csv(
            os.path.join(summary_outpath, subfolder_name, table_csvname)
        )
        group_averages = df.groupby(
            [x_value, replicate_col_name], as_index=False, observed=True
        )[feature]
        # Reset the index to get a clean DataFrame
        # average_df = group_averages.reset_index()
        avg_summary = group_averages.describe()
        avg_summary_sorted = avg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        avg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, feature_csvname)
        )

        # do the agg by passage group only
        group_averages_agg = df.groupby([x_value], as_index=False, observed=True)[
            feature
        ]
        avg_agg_summary = group_averages_agg.describe()
        avg_agg_summary_sorted = avg_agg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        )
        avg_agg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, agg_feature_csvname)
        )
        print(
            f"saved files {(table_csvname, feature_csvname, agg_feature_csvname)} to {summary_outpath}"
        )
        return True
    except ValueError as e:
        print(f"Could not make summary stats: {e}")
        return False
    
summary_outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/Cell_Size_Data/summary_stats/"

for feature in ["Cell_AreaShape_Area","Nuclei_AreaShape_Area","Cell_Nuclei_Area_Ratio"]:
    make_summary_stats_for_df_and_feature(
        combined_cell_df_mitolyso,
        "AllGroups",
        feature,
        summary_outpath,
        df_tag="original",
        feature_name=feature,
        include_cols=[
            "Cell_Number_Object_Number",
            "Cell_AreaShape_Area",
            "Nuclei_AreaShape_Area",
            "Cell_Nuclei_Area_Ratio",
            "Cell_Children_Lysosomes_Count",
            "Cell_Children_Mitochondria_Count",
        ],
    )
    make_summary_stats_for_df_and_feature(
        combined_cell_df_mitolyso_borders_excluded,
        "AllGroups",
        feature,
        summary_outpath,
        df_tag="borders_excluded",
        feature_name=feature,
        include_cols=[
            "Cell_Number_Object_Number",
            "Cell_AreaShape_Area",
            "Nuclei_AreaShape_Area",
            "Cell_Nuclei_Area_Ratio",
            "Cell_Children_Lysosomes_Count",
            "Cell_Children_Mitochondria_Count",
        ],
    )
display(combined_cell_df_mitolyso)

# Make Feature Lists here:

In [ ]:

#file_path = '/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/Cellcsv_columns.txt'


columns_list = define_cell_features(combined_cell_df_mitolyso)

mito_features = make_feature_dict([col for col in columns_list if 'Mito' in col])
lyso_features = make_feature_dict([col for col in columns_list if 'Lysosome' in col or 'LAMP1' in col or 'Lyso' in col])
nuc_features = make_feature_dict([col for col in columns_list if 'Nuc' in col or 'DAPI' in col])
print(columns_list)


In [ ]:
feature_dicts = [mito_features, lyso_features, nuc_features]
feature_names = ["Mitochondria Features", "Lysosome Features", "Nucleus Features"]

# Define the output file path
output_file_path = 'allfeatures_file.md'

# Open the file in write mode
with open(output_file_path, 'w') as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f"- {feature}\n")
            file.write("\n")
            
print(f"List has been written to {output_file_path}")


sns.pairplot(cell_df, hue='Passage Group', vars=mito_features['radialdistribution'], diag_kind='kde', plot_kws={'alpha':0.5})
plt.show()

## Normalize features to control (Passage 6-8)